In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import timeit
import csv
import numba
from numba import njit, prange, get_num_threads, get_thread_id, set_num_threads
from sklearn.datasets import make_regression

import sys

In [2]:
# numba.set_num_threads(get_num_threads() // 2)

In [3]:
current_dir = os.getcwd()
# current_dir

In [4]:
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(current_dir), '../')))

In [5]:
# os.path.dirname(current_dir)

In [6]:
# sys.path

In [7]:
# -----------------------------
# IMPORT YOUR FUNCTIONS
# -----------------------------
from Experiments.Numba.OMP_Implementations.OMP_numba_parallel import (
    OMP_serial,
    OMP_numba_serial,
    OMP_numba_jit,
    OMP_numba_njit,
    OMP_numba_njit_parallel,
    OMP_numba_njit_parallel_pbatches,
)

In [8]:
# import importlib
from Implementations.AK_SVD.Approx_V1_1 import OMP as OMP_A_V1_1

from Implementations.AK_SVD.Numba_Approx_V0 import OMP as OMP_NA_V0
from Implementations.AK_SVD.Numba_Approx_V1 import OMP as OMP_NA_V1
from Implementations.AK_SVD.Numba_Approx_V2 import OMP as OMP_NA_V2

from Implementations.AK_SVD.Para_Numba_Approx_V0 import OMP as OMP_PNA_V0
from Implementations.AK_SVD.Para_Numba_Approx_V1 import OMP as OMP_PNA_V1

In [9]:
# -----------------------------
# DATA SETUP
# -----------------------------
D, y = make_regression(
    n_samples=50,
    n_features=300,
    n_targets=20000,
    noise=4,
    random_state=0
)

Y = y.T.astype(np.float64)
D = D.T.astype(np.float64)

n_samples = Y.shape[0]

In [10]:
# -----------------------------
# PARAMS
# -----------------------------
T_0 = 1
N = 10  # timeit runs

functions = [
    # ("OMP_serial", OMP_serial),
    # ("OMP_numba_serial", OMP_numba_serial),
    # ("OMP_numba_jit", OMP_numba_jit),
    # ("OMP_numba_njit", OMP_numba_njit),
    # ("OMP_numba_njit_parallel", OMP_numba_njit_parallel),
    ("OMP_numba_njit_parallel_pbatches", OMP_numba_njit_parallel_pbatches),
    
    ("Approx_V1_1", OMP_A_V1_1),
    # ("Numba_Approx_V0", OMP_NA_V0),
    # ("Numba_Approx_V1", OMP_NA_V1),  
    ("Numba_Approx_V2", OMP_NA_V2),   
    # ("Para_Numba_Approx_V0", OMP_PNA_V0),   
    ("Para_Numba_Approx_V1", OMP_PNA_V1),   
]

max_threads = numba.get_num_threads()

thread_list = [1, 3, 6, 9, 12]
batch_sizes = [2**i for i in range(1, int(np.log2(n_samples/16)) + 1)]
batch_sizes = [b for b in batch_sizes if b <= n_samples]

print("Threads:", thread_list)
print("Batch sizes:", batch_sizes)

Threads: [1, 3, 6, 9, 12]
Batch sizes: [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]


In [11]:
# -----------------------------
# WARMUP (compile all functions)
# -----------------------------
for name, fn in functions:
    print(name)
    try:
        fn(Y, T_0, D, batch_size=2, float_dtype=np.float64)
    except Exception as e:
        print(e)
        try:
            fn(Y, T_0, D, batch_size=2)
        except Exception as f:
            print(f)
        print()

OMP_numba_njit_parallel_pbatches
some keyword arguments unexpected


C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:1039: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]
C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:1039: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]



Approx_V1_1
Numba_Approx_V2
Para_Numba_Approx_V1


In [12]:
# -----------------------------
# BENCHMARK
# -----------------------------
results = []

for name, fn in functions:
    print(f"\n--- {name} ---")
    
    for n_threads in thread_list:
        set_num_threads(n_threads)
        
        for batch_size in batch_sizes:
            # define callable for timeit
            stmt = lambda: fn(Y, T_0, D, batch_size=batch_size)
            stmt2 = lambda: fn(Y, T_0, D, batch_size=batch_size, float_dtype=np.float64)
            
            
            try:
                stmt2()
            except:
                try:
                    stmt()
                except:
                    print(f"FAIL: {name}, threads={n_threads}, batch={batch_size}")
                    continue
            try:
                t = timeit.timeit(stmt2, number=N) / N
            except Exception as e:
                try:
                    t = timeit.timeit(stmt, number=N) / N               
                except:
                    print(f"FAIL: {name}, threads={n_threads}, batch={batch_size}")
                    continue

            print(f"{name} | threads={n_threads} | batch={batch_size} | {t:.6f}s")

            results.append((name, n_threads, batch_size, t))


--- OMP_numba_njit_parallel_pbatches ---
OMP_numba_njit_parallel_pbatches | threads=1 | batch=2 | 0.159020s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=4 | 0.142004s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=8 | 0.128551s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=16 | 0.116596s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=32 | 0.113132s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=64 | 0.116651s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=128 | 0.113628s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=256 | 0.107616s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=512 | 0.166651s
OMP_numba_njit_parallel_pbatches | threads=1 | batch=1024 | 0.164403s
OMP_numba_njit_parallel_pbatches | threads=3 | batch=2 | 0.109774s
OMP_numba_njit_parallel_pbatches | threads=3 | batch=4 | 0.102036s
OMP_numba_njit_parallel_pbatches | threads=3 | batch=8 | 0.107401s
OMP_numba_njit_parallel_pbatches | threads=3 | batch=16 | 0.087071s
OMP_num

In [13]:
# -----------------------------
# SAVE CSV
# -----------------------------
file_name = 'thread_scaling.csv'
with open(file_name, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["function", "threads", "batch_size", "time_sec"])
    writer.writerows(results)

print(f"\nSaved to {file_name}")


Saved to thread_scaling.csv
